# Hard Case: Elements inside a `<table>`

When the data lives in an **HTML table** (`<table>`), there's no need to grind through it
row by row with BeautifulSoup. There's a **magic one-liner**: `pandas.read_html()`.

`pd.read_html(...)` reads **all** `<table>` elements on the page and returns a **list of
DataFrames** — already tidy, you just pick the table you want.

> Needs an HTML parser: `lxml` or `html5lib` (this project already has `lxml`).

**Tooling:** `pandas` (+ `lxml`). BeautifulSoup is used only for comparison.


## 1. Read tables from HTML

We use HTML containing **two** tables (products & exchange rates) to show that `read_html`
returns a **list** — one DataFrame per table.

> pandas 3.x note: wrap the HTML string with `io.StringIO(...)` when passing it directly.


In [1]:
from io import StringIO

import pandas as pd

HTML_TABLE = """
<h3>Product List</h3>
<table>
  <thead><tr><th>Name</th><th>Price</th><th>Stock</th></tr></thead>
  <tbody>
    <tr><td>Learn Python</td><td>Rp75.000</td><td>In Stock</td></tr>
    <tr><td>Wireless Mouse</td><td>Rp150.000</td><td>In Stock</td></tr>
    <tr><td>Mechanical Keyboard</td><td>Rp350.000</td><td>Out of Stock</td></tr>
  </tbody>
</table>

<h3>Exchange Rates</h3>
<table>
  <thead><tr><th>Currency</th><th>Value (Rp)</th></tr></thead>
  <tbody>
    <tr><td>USD</td><td>16.000</td></tr>
    <tr><td>EUR</td><td>17.500</td></tr>
  </tbody>
</table>
"""

# read_html -> list of DataFrames (one per <table>)
# flavor="lxml" so it uses the lxml parser (no html5lib needed)
tables = pd.read_html(StringIO(HTML_TABLE), flavor="lxml")
print("Number of tables found:", len(tables))
tables[0]


Number of tables found: 2


,Name,Price,Stock
0,Learn Python,Rp75.000,In Stock
1,Wireless Mouse,Rp150.000,In Stock
2,Mechanical Keyboard,Rp350.000,Out of Stock


## 2. Select a table with `match=`, then clean the columns

When there are many tables, use `match="keyword"` to pick the table that **contains
specific text**. Once you have the DataFrame, clean the columns as usual with pandas.

**Reading from a real URL** (e.g. Wikipedia):

```python
# the shortest way (if the site allows it):
tables = pd.read_html("https://en.wikipedia.org/wiki/List_of_countries_by_population", match="Population")

# if the site needs headers / blocks you, fetch it first with requests, then StringIO:
import requests
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10).text
tables = pd.read_html(StringIO(html), match="Population")
```


In [2]:
# select the table containing the word "Price" (the products table)
products = pd.read_html(StringIO(HTML_TABLE), match="Price", flavor="lxml")[0]
print("Before cleaning:")
print(products, "\n")

# clean the Price column: "Rp75.000" -> 75000 (int). The dot is a thousands separator.
products["Price"] = products["Price"].str.replace(r"[^0-9]", "", regex=True).astype(int)

print("After cleaning:")
print(products)
print("\nTotal inventory value (sum of prices):", products["Price"].sum())


Before cleaning:
                  Name      Price         Stock
0         Learn Python   Rp75.000      In Stock
1       Wireless Mouse  Rp150.000      In Stock
2  Mechanical Keyboard  Rp350.000  Out of Stock 

After cleaning:
                  Name   Price         Stock
0         Learn Python   75000      In Stock
1       Wireless Mouse  150000      In Stock
2  Mechanical Keyboard  350000  Out of Stock

Total inventory value (sum of prices): 575000


## 3. Comparison: parsing the table manually with BeautifulSoup

For comparison, here's the manual way with BeautifulSoup. The result is the same, but the code
is much longer. For tables, **`pd.read_html` is almost always better**.

Manual BS4 only wins when the structure is "weird" (not a standard `<table>`, or you need to
grab attributes like `href` inside a cell).

## Exercise
Take the **exchange-rate** table (use `match="Currency"` or index `[1]`), then convert the value
column to numbers and compute its average.

> Note: `match` searches for text **inside** the `<table>`. The word "Exchange" is in the `<h3>`
> (outside the table), so it can't be used for `match` — use a column header instead.


In [3]:
from bs4 import BeautifulSoup

# --- Comparison: manual BS4 ---
soup = BeautifulSoup(HTML_TABLE, "html.parser")
product_table = soup.find("table")
rows = []
for tr in product_table.find("tbody").find_all("tr"):
    rows.append([td.get_text(strip=True) for td in tr.find_all("td")])
print("Manual BS4 parsing result:")
for row in rows:
    print("  ", row)

# --- Sample solution to the exercise: exchange-rate table ---
# match searches for text INSIDE the <table>; "Exchange" only appears in the <h3> (outside the table),
# so we match on a column header that lives inside the table: "Currency".
# thousands="." -> "16.000" is read as 16000 (the dot is a thousands separator, not a decimal point).
rates = pd.read_html(StringIO(HTML_TABLE), match="Currency", flavor="lxml", thousands=".")[0]
print("\nExchange-rate table:")
print(rates)
print("Value column dtype:", rates["Value (Rp)"].dtype)
print("Average value:", rates["Value (Rp)"].mean())


Manual BS4 parsing result:
   ['Learn Python', 'Rp75.000', 'In Stock']
   ['Wireless Mouse', 'Rp150.000', 'In Stock']
   ['Mechanical Keyboard', 'Rp350.000', 'Out of Stock']

Exchange-rate table:
  Currency  Value (Rp)
0      USD       16000
1      EUR       17500
Value column dtype: int64
Average value: 16750.0
